# Olist order lines — JSONL

Real records copied from our replayed Odoo/Olist landing zone. `clean` retains
the source values; `drift` contains clearly marked teaching copies with intentional errors.
Original landing files are untouched. Source IDs and hashes are in `datasets/olist-samples.json`.

Follow the cells: configure Spark → declare the schema → read with ordinary Spark
→ explore the data → optionally apply schema drift. The ordinary reader and
exploration work without importing `schema_drift.py`.


In [ ]:
import os
import json
import uuid
from pathlib import Path
from functools import reduce
from pyspark.sql import SparkSession, functions as F, types as T

WORK = Path.cwd().resolve()  # Start Jupyter in the schema-drifting directory.
DATA_ROOT = Path(os.environ.get("STUDENT_DATA_ROOT", str(WORK / "datasets"))).resolve()
OUTPUT_ROOT = Path(
    os.environ.get("STUDENT_OUTPUT_ROOT", str(WORK / "student-output"))
).resolve()
DATASET_NAME = "olist_order_lines"
SOURCE_DIR = DATA_ROOT / "order_lines" / "olist_complex"
SOURCE_PATH = os.environ.get("STUDENT_SOURCE_PATH", SOURCE_DIR.as_uri())
run_id = str(uuid.uuid4())
batch_id = DATASET_NAME + "_" + run_id

# Set False to skip schema drift; the reader and exploration still run.
RUN_SCHEMA_DRIFT = os.environ.get("STUDENT_SCHEMA_DRIFT", "1") == "1"
RUN_CORRUPT_RECORDS = os.environ.get("STUDENT_CORRUPT_RECORDS", "1") == "1"
RUN_EXPLORATION = os.environ.get("STUDENT_EXPLORATION", "1") == "1"
RUN_GREAT_EXPECTATIONS = os.environ.get("STUDENT_GREAT_EXPECTATIONS", "1") == "1"

builder = (
    SparkSession.builder.appName(DATASET_NAME)
    .master(os.environ.get("SPARK_MASTER", "local[2]"))
    .config("spark.ui.enabled", "false")
    .config("spark.sql.shuffle.partitions", "2")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.catalog.datalake", "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.datalake.type", "hadoop")
    .config(
        "spark.sql.catalog.datalake.warehouse", (OUTPUT_ROOT / "warehouse").as_uri()
    )
)
if os.environ.get("ICEBERG_SPARK_RUNTIME_JAR"):
    builder = builder.config("spark.jars", os.environ["ICEBERG_SPARK_RUNTIME_JAR"])
spark = builder.getOrCreate()
spark.sparkContext.setLogLevel("WARN")


## The target schema

Odoo exports a relationship as `[id, name]`, or `false` when absent.
We represent a present relationship as a struct. Only the primary ID is
required at the top level. Monetary values use decimals; dates use timestamps.


In [ ]:
relationship_schema = T.StructType(
    [
        T.StructField("id", T.LongType(), False),
        T.StructField("name", T.StringType(), False),
    ]
)

expected_schema = T.StructType(
    [
        T.StructField("id", T.LongType(), False),
        T.StructField("order_id", relationship_schema, True),
        T.StructField("product_id", relationship_schema, True),
        T.StructField("product_uom_qty", T.DecimalType(38, 18), True),
        T.StructField("price_unit", T.DecimalType(38, 18), True),
        T.StructField("discount", T.DecimalType(38, 18), True),
        T.StructField("price_subtotal", T.DecimalType(38, 18), True),
        T.StructField("create_date", T.TimestampType(), True),
        T.StructField("write_date", T.TimestampType(), True),
        T.StructField("qty_delivered", T.DecimalType(38, 18), True),
        T.StructField("qty_invoiced", T.DecimalType(38, 18), True),
    ]
)
expected_schema.simpleString()


## Read with ordinary Spark — no schema-drift module

This section makes the file encoding explicit and produces a DataFrame.
Invalid values may become null here; reading alone does not validate or quarantine them.
The optional profiler cells report findings without rejecting records.


In [ ]:
# Read original lines. Extracting field text avoids mixed [number, name]
# arrays and false sentinels forcing an unsuitable inferred schema.
raw_lines = spark.read.text(SOURCE_PATH).withColumn("_source_file", F.input_file_name())
field_names = [
    name for name in spark.read.json(SOURCE_PATH).columns if name != "_corrupt_record"
]
records = raw_lines.select(
    *[F.get_json_object("value", "$." + name).alias(name) for name in field_names],
    F.col("value").alias("_raw_record"),
    F.col("_source_file"),
)

landing_df = records  # Preserve original values before preview conversions.

# Convert only from strings: try_cast returns null for invalid values.
for field in expected_schema:
    if isinstance(field.dataType, T.StructType):
        records = records.withColumn(
            "_pair", F.from_json(F.col(field.name), T.ArrayType(T.StringType()))
        )
        relationship = F.struct(
            F.expr("try_cast(element_at(_pair, 1) as bigint)").alias("id"),
            F.element_at("_pair", 2).alias("name"),
        )
        records = records.withColumn(
            field.name, F.when(F.size("_pair") == 2, relationship)
        ).drop("_pair")
    elif isinstance(field.dataType, T.StringType):
        records = records.withColumn(
            field.name,
            F.when(F.lower(F.col(field.name)) == "false", None).otherwise(
                F.col(field.name)
            ),
        )
    else:
        records = records.withColumn(
            field.name,
            F.expr(
                "try_cast(`"
                + field.name
                + "` as "
                + field.dataType.simpleString()
                + ")"
            ),
        )

records.printSchema()
records.show(6, truncate=60)

example = json.loads((DATA_ROOT / "olist-samples.json").read_text())["order_lines"]
assert records.count() == example["expected_rows"]
assert (
    records.where(F.col("id") == example["source_ids"][0])
    .where(F.col("product_id.id").isNotNull())
    .limit(1)
    .count()
    == 1
)


In [ ]:
# Parse original text with a source representation schema. Relationships
# are mixed [id, name] arrays; Spark represents their elements as strings.
parser_fields = [
    T.StructField(
        field.name,
        (
            T.ArrayType(T.StringType())
            if isinstance(field.dataType, T.StructType)
            else field.dataType
        ),
        True,
    )
    for field in expected_schema
]
parser_schema = T.StructType(
    parser_fields + [T.StructField("_corrupt_record", T.StringType())]
)
parsed_df = raw_lines.select(
    F.from_json("value", parser_schema, {"mode": "PERMISSIVE"}).alias("parsed"),
    "_source_file",
).select("parsed.*", "_source_file")


## Independent profilers

Enable any combination using the four flags, or comment out an entire cell.
Each profiler writes audit findings only. None repairs or routes records.
Parser corruption, schema differences, statistical measures and business
expectations answer different questions. CSV string reads do not detect
numeric conversion failures; the typed preview and GX rules expose nulls.
Parquet normally has no `_corrupt_record`; that absence is audited explicitly.
Spark's corrupt record contains raw text, not a precise failure reason.


In [ ]:
if RUN_CORRUPT_RECORDS:
    from data_profilers.schema_drift import profile_corrupt_records

    profile_corrupt_records(parsed_df, dataset_name=DATASET_NAME, run_id=run_id)
    spark.table("datalake.operations.corrupt_records").where(
        F.col("run_id") == run_id
    ).show(truncate=100)


In [ ]:
if RUN_SCHEMA_DRIFT:
    from data_profilers.schema_drift import profile_schema_drift

    profile_schema_drift(
        spark,
        SOURCE_PATH,
        expected_schema,
        source_format="jsonl",
        dataset_name=DATASET_NAME,
        run_id=run_id,
    )
    findings = spark.table("datalake.operations.schema_drift_profiles").where(
        F.col("run_id") == run_id
    )
    findings.select("source_file", "status", "details_json").show(truncate=120)
    assert findings.count() > 0
    assert findings.where(F.col("status") == "INSPECTION_FAILED").count() == 0


In [ ]:
if RUN_EXPLORATION:
    from data_profilers.exploration import explore_data, write_exploration_to_iceberg

    measurements = explore_data(
        spark,
        records,
        dataset_name=DATASET_NAME,
        run_id=run_id,
        batch_id=batch_id,
        columns=expected_schema.names,
        required_columns=["id"],
        processing_stage="LANDING",
    )
    table = write_exploration_to_iceberg(spark, measurements)
    spark.table(table).where(F.col("run_id") == run_id).show(12, truncate=False)


In [ ]:
if RUN_GREAT_EXPECTATIONS:
    import great_expectations as gx
    from data_profilers.great_expectations import profile_expectations

    # Every sample contains a deliberately missing or invalid primary ID.
    # These failures are expected teaching findings, not pipeline crashes.
    rules = [
        gx.expectations.ExpectColumnValuesToNotBeNull(column="id"),
        gx.expectations.ExpectColumnValuesToBeBetween(column="id", min_value=1),
        gx.expectations.ExpectTableRowCountToBeBetween(min_value=1),
    ]
    profile_expectations(records, rules, dataset_name=DATASET_NAME, run_id=run_id)
    results = spark.table("datalake.operations.expectation_profiles").where(
        F.col("run_id") == run_id
    )
    results.select("status", "details_json").show(truncate=120)
    assert results.where(F.col("status") == "FAILED").count() == 1
    assert results.where(F.col("status") == "RULE_PASSED").count() >= 1


## Student exercises

Run each profiler alone. Compare parser failures with required-ID failures.
Explain why an Odoo relationship array differs from the target struct.
Inspect the nested fields and source-file paths in the audit tables.
Writing business data is a separate student decision, for example:
`records.writeTo("datalake.bronze.orders").using("iceberg").create()`.
No profiler produces Bronze or quarantine tables automatically.


In [ ]:
spark.stop()
print("PASS: ordinary reader and independently optional profilers")
